# `nyfed(series, start, end=None)` and `nyfed_series()`

New York Fed series from `indicators.nyfed`, oldest first: the five overnight reference rates (SOFR, TGCR, BGCR, EFFR, OBFR), daily, and the Fed's portfolio (the System Open Market Account, SOMA) by holding, weekly. `nyfed_series()` is the map from each series id to its dataset, name, units, frequency and category. Every cell runs against prod, read-only, as `readonly_user`. Change the arguments and re-run any cell.

A series is named by its New York Fed id (`sofr`, `total`) or by its name, in any case and spacing. Periods with no value are left out, never filled.

> **About the data shown.** Source: Federal Reserve Bank of New York.
>
> The SOFR, TGCR, BGCR, EFFR and OBFR data is subject to the Terms of Use posted at newyorkfed.org. The New York Fed is not responsible for publication of the SOFR, TGCR, BGCR, EFFR and OBFR data by arora_data, does not sanction or endorse any particular republication, and has no liability for your use.
>
> The Secured Overnight Financing Rate (SOFR) Data and Broad General Collateral Rate (BGCR) Data are calculated using data provided under a license granted to the New York Fed by DTCC Solutions LLC (“Solutions”), an affiliate of The Depository Trust & Clearing Corporation. Solutions, its affiliates, and third parties from which they obtained data have no liability for the content of this material.
>
> SOMA holdings: © 2026 Federal Reserve Bank of New York. Content from the New York Fed subject to the Terms of Use at newyorkfed.org.
>
> Modifications, by arora_data, not the New York Fed: the SOMA amounts are the New York Fed's dollar figures divided by 1,000,000, so they are in USD millions; the joined rates table and the year-end totals in USD trillions are this notebook's own views. The New York Fed's data shown here stays under the New York Fed's Terms of Use, not this repository's terms.
>
> arora_data is not affiliated with the New York Fed. The New York Fed does not sanction, endorse, or recommend any products or services offered by arora_data. None of this is investment advice.

In [1]:
from datetime import date, timedelta

import polars as pl
from IPython.display import Markdown

from arora_data.dates import today_et
from arora_data.frames import to_df
from arora_data.nyfed import NyfedValue
from arora_data.sync import nyfed, nyfed_series


def show(rows):
    """One result or a list of results as a Markdown table."""
    rows = [r.model_dump() for r in (rows if isinstance(rows, list) else [rows])]
    if not rows:
        return Markdown("0 rows")
    head = list(rows[0])
    lines = ["|".join(head), "|".join("---" for _ in head)]
    lines += ["|".join(str(r[c]).replace("|", "/") for c in head) for r in rows]
    return Markdown(f"{len(rows)} rows\n\n" + "\n".join(lines))

## The series
Every series the table holds, read from it on every call. The labels come from each series' latest row; `first_date` and `latest_date` are its first and latest dates with a value.

In [2]:
show(nyfed_series())

13 rows

series_id|dataset|name|units|frequency|category|first_date|latest_date
---|---|---|---|---|---|---|---
agencies|nyfed/soma_summary|SOMA agency debt|USD millions|weekly|Liquidity — SOMA|2003-07-09|2026-11-18
bgcr|nyfed/reference_rates|Broad General Collateral Rate (BGCR)|percent|daily|Money Markets — Rates|2025-06-25|2026-11-19
bills|nyfed/soma_summary|SOMA Treasury bills|USD millions|weekly|Liquidity — SOMA|2003-07-09|2026-11-18
cmbs|nyfed/soma_summary|SOMA agency CMBS|USD millions|weekly|Liquidity — SOMA|2003-07-09|2026-11-18
effr|nyfed/reference_rates|Effective Federal Funds Rate (EFFR)|percent|daily|Money Markets — Rates|2025-06-26|2026-11-19
frn|nyfed/soma_summary|SOMA floating-rate notes|USD millions|weekly|Liquidity — SOMA|2014-05-21|2026-11-18
mbs|nyfed/soma_summary|SOMA agency MBS|USD millions|weekly|Liquidity — SOMA|2003-07-09|2026-11-18
notesbonds|nyfed/soma_summary|SOMA Treasury notes & bonds|USD millions|weekly|Liquidity — SOMA|2003-07-09|2026-11-18
obfr|nyfed/reference_rates|Overnight Bank Funding Rate (OBFR)|percent|daily|Money Markets — Rates|2025-06-26|2026-11-19
sofr|nyfed/reference_rates|Secured Overnight Financing Rate (SOFR)|percent|daily|Money Markets — Rates|2025-06-25|2026-11-19
tgcr|nyfed/reference_rates|Tri-Party General Collateral Rate (TGCR)|percent|daily|Money Markets — Rates|2025-06-25|2026-11-19
tips|nyfed/soma_summary|SOMA TIPS|USD millions|weekly|Liquidity — SOMA|2003-07-09|2026-11-18
total|nyfed/soma_summary|SOMA total holdings (par)|USD millions|weekly|Liquidity — SOMA|2003-07-09|2026-11-18

## One series, by id or by name
SOFR over the last two weeks. The id and the name, in any case and spacing, give the same rows. Rates are in percent: 3.91 is 3.91%.

In [3]:
two_weeks = today_et() - timedelta(days=14)
assert nyfed("sofr", two_weeks) == nyfed("SOFR", two_weeks) == nyfed(" secured overnight FINANCING rate (sofr) ", two_weeks)
show(nyfed("sofr", two_weeks))

8 rows

series_id|name|date|value|units
---|---|---|---|---
sofr|Secured Overnight Financing Rate (SOFR)|2026-11-09|3.88000000|percent
sofr|Secured Overnight Financing Rate (SOFR)|2026-11-10|3.89000000|percent
sofr|Secured Overnight Financing Rate (SOFR)|2026-11-12|3.90000000|percent
sofr|Secured Overnight Financing Rate (SOFR)|2026-11-13|3.90000000|percent
sofr|Secured Overnight Financing Rate (SOFR)|2026-11-16|3.93000000|percent
sofr|Secured Overnight Financing Rate (SOFR)|2026-11-17|3.92000000|percent
sofr|Secured Overnight Financing Rate (SOFR)|2026-11-18|3.90000000|percent
sofr|Secured Overnight Financing Rate (SOFR)|2026-11-19|3.91000000|percent

## The five rates side by side
One frame per rate, joined on the date. The secured rates (SOFR, TGCR, BGCR) are repo rates; EFFR and OBFR are unsecured.

In [4]:
rates = ["sofr", "tgcr", "bgcr", "effr", "obfr"]
frames = [to_df(nyfed(r, two_weeks), NyfedValue).select("date", pl.col("value").alias(r)) for r in rates]
wide = frames[0]
for f in frames[1:]:
    wide = wide.join(f, on="date", how="full", coalesce=True)
wide.sort("date")

date,sofr,tgcr,bgcr,effr,obfr
date,"decimal[38,8]","decimal[38,8]","decimal[38,8]","decimal[38,8]","decimal[38,8]"
2026-11-09,3.88000000,3.86000000,3.86000000,3.88000000,3.88000000
2026-11-10,3.89000000,3.87000000,3.87000000,3.88000000,3.88000000
2026-11-12,3.90000000,3.87000000,3.87000000,3.88000000,3.88000000
2026-11-13,3.90000000,3.87000000,3.87000000,3.88000000,3.88000000
2026-11-16,3.93000000,3.90000000,3.90000000,3.88000000,3.88000000
2026-11-17,3.92000000,3.90000000,3.90000000,3.88000000,3.88000000
2026-11-18,3.90000000,3.88000000,3.88000000,3.88000000,3.88000000
2026-11-19,3.91000000,3.88000000,3.88000000,3.88000000,3.88000000


## Which days have a rate
Holidays have no row. The secured rates also skip days only the bond market is closed: Good Friday 2026-04-03 has EFFR and OBFR but not SOFR.

In [5]:
good_friday = date(2026, 4, 3)
print({r: [str(v.date) for v in nyfed(r, good_friday, good_friday)] for r in rates})

{'sofr': [], 'tgcr': [], 'bgcr': [], 'effr': ['2026-04-03'], 'obfr': ['2026-04-03']}


## Short rate history
The main repo fetches only each rate's latest 250 days, so the stored history starts in June 2025 and grows from there. EFFR's longer history is `fred("DFF")`.

In [6]:
print({s.series_id: str(s.first_date) for s in nyfed_series() if s.dataset == "nyfed/reference_rates"})

{'bgcr': '2025-06-25', 'effr': '2025-06-26', 'obfr': '2025-06-26', 'sofr': '2025-06-25', 'tgcr': '2025-06-25'}


## The Fed's portfolio
SOMA holdings at par, in USD millions, weekly from July 2003. Each week is dated at the summary's as-of day, a Wednesday or the day next to it around a holiday. The year-end total, in USD trillions:

In [7]:
total = to_df(nyfed("total", date(1900, 1, 1)), NyfedValue)
print(f"{total.height:,} weeks, {total['date'].min()} to {total['date'].max()}")
(total.group_by(pl.col("date").dt.year().alias("year")).agg(pl.col("value").sort_by("date").last())
      .with_columns((pl.col("value") / 1_000_000).round(2).alias("USD trillions")).drop("value").sort("year"))

1,220 weeks, 2003-07-09 to 2026-11-18


year,USD trillions
i32,"decimal[38,8]"
2003,0.67000000
2004,0.72000000
2005,0.74000000
2006,0.77000000
2007,0.75000000
…,…
2022,8.04000000
2023,7.11000000
2024,6.41000000


## The parts add up to the total
The seven holdings sum to `total` in every week, to within a few dollars.

In [8]:
soma = [s for s in nyfed_series() if s.dataset == "nyfed/soma_summary"]
week = max(s.latest_date for s in soma)
held = {s.series_id: nyfed(s.series_id, week, week)[0].value for s in soma}
for k, v in held.items():
    print(f"{k:<11} {v:>20}")
print("parts - total:", sum(v for k, v in held.items() if k != "total") - held["total"])

agencies           2347.00000000
bills            580659.92670000
cmbs               7311.56132540
frn               22873.04710000
mbs             1876153.37377930
notesbonds      3597864.73450000
tips             277655.58590000
total           6364865.22930470
parts - total: 0E-8


## Gaps
A week the summary leaves a holding blank is left out, never filled. The Fed held no floating-rate notes before 2014-05-21, and the agency debt holding is blank from 2003-12-24 to 2008-09-17.

In [9]:
print([str(v.date) for v in nyfed("frn", date(2014, 4, 1), date(2014, 6, 30))])
print([(str(v.date), v.value) for v in nyfed("agencies", date(2003, 11, 26), date(2008, 10, 1))])

['2014-05-21', '2014-05-28', '2014-06-04', '2014-06-11', '2014-06-18', '2014-06-25']


[('2003-11-26', Decimal('10.00000000')), ('2003-12-03', Decimal('10.00000000')), ('2003-12-10', Decimal('0E-8')), ('2003-12-17', Decimal('0E-8')), ('2008-09-24', Decimal('10000.00000000')), ('2008-10-01', Decimal('14500.00000000'))]


## Revisions
The main repo re-reads both datasets on every run, so a revised value replaces the earlier one, and earlier values are not stored.

## Bad input
An unknown id or name, or a start after the end, raises a clear error before any values are read.

In [10]:
for args in [("DFF", date(2026, 1, 1)), ("secured overnight financing rate", date(2026, 1, 1)), (None, date(2026, 1, 1)),
             ("sofr", date(2026, 1, 2), date(2026, 1, 1))]:
    try:
        nyfed(*args)
    except ValueError as e:
        print(e)

'DFF' is not a stored New York Fed series: use an id or name from nyfed_series(), such as sofr or 'SOMA total holdings (par)'


'secured overnight financing rate' is not a stored New York Fed series: use an id or name from nyfed_series(), such as sofr or 'SOMA total holdings (par)'


None is not a stored New York Fed series: use an id or name from nyfed_series(), such as sofr or 'SOMA total holdings (par)'
start must be on or before end
